In [1]:
import pandas as pd
import numpy as np
import os

def clean_hts_description(desc):
    if not isinstance(desc, str):
        return ""
    desc = str(desc).strip()
    if desc.endswith(':'):
        desc = desc[:-1]
    return desc.strip()

def process_hts_barrier_logic():
    input_path = "../data/raw/htsdata.csv"
    output_path = "../data/intermediate/diez_digitos.csv"
    
    print(f"Procesando {input_path}...")
    try:
        # Leemos todo como string para no perder ceros ni formatos
        df = pd.read_csv(input_path, dtype=str)
    except FileNotFoundError:
        print("Error: No se encontró el archivo.")
        return

    final_rows = []
    
    # La Pila (Stack) guarda: { Indent: { 'desc': texto, 'code_len': longitud_codigo } }
    hierarchy = {}

    for index, row in df.iterrows():
        # --- 1. PARSEO DE DATOS ---
        
        # Indent
        try:
            raw_indent = row['Indent']
            if pd.isna(raw_indent) or str(raw_indent).strip() == '':
                continue
            indent = int(float(str(raw_indent)))
        except ValueError:
            continue

        # Código HTS
        raw_code = str(row['HTS Number']) if pd.notna(row['HTS Number']) else ""
        clean_code = raw_code.replace(".", "").strip()
        if clean_code.lower() == 'nan': clean_code = ""
        code_len = len(clean_code)

        # Descripción
        raw_desc = clean_hts_description(row['Description'])
        
        # Tasa General (Para el filtro nuevo)
        general_rate = str(row['General Rate of Duty']).strip() if pd.notna(row['General Rate of Duty']) else ""

        # --- 2. GESTIÓN DE LA PILA (MANTENER JERARQUÍA) ---
        
        # Borramos cualquier nivel más profundo o igual al actual (Reseteo de rama)
        levels_to_remove = [k for k in hierarchy.keys() if k >= indent]
        for k in levels_to_remove:
            del hierarchy[k]
            
        # Agregamos el nodo actual a la pila
        hierarchy[indent] = {
            'desc': raw_desc,
            'code_len': code_len
        }

        # --- 3. PROCESAMIENTO DE CÓDIGOS DE 10 DÍGITOS ---
        
        if code_len == 10:
            # CONDICIÓN NUEVA: Si tiene Tasa General, SE DESCARTA.
            if general_rate != "":
                continue

            # CONSTRUCCIÓN DE DESCRIPCIÓN (MIRAR HACIA ATRÁS)
            desc_parts = []
            
            # Empezamos por el nivel actual (el producto mismo) y vamos subiendo (Indent - 1, Indent - 2...)
            # Usaremos un bucle hacia atrás para detectar la "Barrera"
            
            # 1. Agregamos la descripción propia
            if hierarchy[indent]['desc']:
                desc_parts.insert(0, hierarchy[indent]['desc']) # Insertamos al principio
            
            # 2. Buscamos ancestros hacia arriba
            current_search_indent = indent - 1
            
            while current_search_indent >= 0:
                if current_search_indent in hierarchy:
                    node = hierarchy[current_search_indent]
                    node_code_len = node['code_len']
                    
                    # ¿ES UNA BARRERA? (Código de 4, 6 u 8 dígitos)
                    # Si encontramos uno de estos, PARAMOS de buscar hacia arriba.
                    # NO agregamos su descripción, solo paramos.
                    if node_code_len in [4, 6, 8]:
                        break 
                    
                    # Si NO es barrera (es decir, código vacío/intermedio), LO AGREGAMOS.
                    if node['desc']:
                        desc_parts.insert(0, node['desc']) # Prepend (poner antes de lo que ya tenemos)
                
                # Seguimos subiendo
                current_search_indent -= 1
            
            # Unimos las partes
            full_desc = ". ".join(desc_parts) + "."
            
            final_rows.append({
                'HTS Code': clean_code,
                'Description': full_desc
            })

    # --- 4. EXPORTACIÓN ---
    if final_rows:
        out_df = pd.DataFrame(final_rows)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        out_df.to_csv(output_path, index=False)
        print(f"Proceso finalizado. Se generaron {len(out_df)} registros limpios.")
        print(f"Archivo guardado en: {output_path}")
        print("\nEjemplo de salida:")
        print(out_df.head())
    else:
        print("No se encontraron registros que cumplan con los criterios.")

if __name__ == "__main__":
    process_hts_barrier_logic()

Procesando ../data/raw/htsdata.csv...
Proceso finalizado. Se generaron 11598 registros limpios.
Archivo guardado en: ../data/intermediate/diez_digitos.csv

Ejemplo de salida:
     HTS Code                        Description
0  0101210010                             Males.
1  0101210020                           Females.
2  0101290010  Imported for immediate slaughter.
3  0101290090                             Other.
4  0102210010                       Dairy. Male.
